# 01 — Descarga y extracción de microdatos de mortalidad (INEGI)

**Proyecto:** chihuahua-suicide-seasonality-replication
**Etapa:** Diagnóstico inicial / preparación de datos crudos

Este notebook:
1. Descarga los archivos de microdatos de "Estadísticas de Defunciones Registradas" de INEGI.
2. Extrae los archivos comprimidos (.zip).
3. Carga cada archivo en un DataFrame de pandas (soporta CSV y DBF).
4. Separa los archivos consolidados (que traen varios años juntos) en años individuales, usando la columna de año que encontremos en los datos.
5. Guarda cada año como parquet en `data/interim/`, sin filtrar por causa todavía (el filtro a suicidio, X60-X84, se hace en el notebook 02 de limpieza, de forma documentada).

## ⚠️ Estructura real de los archivos de INEGI (no es un archivo por año)

INEGI agrupa varios años en un mismo archivo para ciertos periodos, y a partir de 2020-2021 entrega un archivo por año. La tabla real es:

| Clave interna | Años que contiene | Uso |
|---|---|---|
| `2005_2009` | 2005-2009 | Solo usamos 2008-2009 para la réplica |
| `2010_2014` | 2010-2014 | Réplica completa |
| `2015_2019` | 2015-2019 | Réplica (2015-2018) + 2019 para extensión |
| `2020` | 2020 | Extensión |
| `2021` | 2021 | Extensión |
| `2022` | 2022 | Extensión |
| `2023` | 2023 | Extensión |
| `2024` | 2024 | Extensión |

Es decir: **8 descargas en total** cubren tanto la réplica (2008-2018) como la extensión (hasta el año más reciente disponible).

⚠️ **Antes de correr este notebook necesitas completar `FILE_GROUPS`** (celda 2) con el enlace directo de descarga de cada uno de estos 8 archivos. INEGI no usa una URL predecible, así que hay que obtenerlas a mano:

1. Ve a https://www.inegi.org.mx/programas/edr/ → pestaña **Microdatos** (o el portal de "Descarga masiva": https://www.inegi.org.mx/app/descarga/default.html, tema = Mortalidad).
2. Para cada uno de los 8 archivos de la tabla, da clic derecho sobre el botón de descarga → "Copiar enlace" (o "Copy link address").
3. Pega cada enlace en el diccionario `FILE_GROUPS` de la celda siguiente.
4. Haz lo mismo para el diccionario de datos (codebook) de cada archivo si quieres guardarlo también (opcional, pero muy recomendado, ya que los códigos pueden cambiar entre periodos consolidados).

Este notebook está escrito para correr **en tu computadora** (no en este entorno de Claude), ya que aquí no tengo acceso de red a inegi.org.mx.

In [ ]:
import os
import io
import zipfile
import requests
import pandas as pd
from pathlib import Path

# --- Rutas del proyecto (ajusta si corres el notebook desde otra ubicación) ---
PROJECT_ROOT = Path("..")  # este notebook vive en notebooks/, el proyecto está un nivel arriba
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "inegi_mortalidad"
RAW_ZIPS_DIR = RAW_DIR / "zips"
RAW_EXTRACTED_DIR = RAW_DIR / "extracted"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

for d in [RAW_ZIPS_DIR, RAW_EXTRACTED_DIR, INTERIM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Carpetas listas:")
print(RAW_ZIPS_DIR)
print(RAW_EXTRACTED_DIR)
print(INTERIM_DIR)

In [ ]:
# --- URLs de descarga (confirmadas por el usuario) ---
# "years" = lista de años que ese archivo contiene (para poder separarlos después).

FILE_GROUPS = {
    "2005_2009": {"years": [2005, 2006, 2007, 2008, 2009], "url": "https://www.inegi.org.mx/contenidos/programas/edr/microdatos/defunciones/datos/defunciones_generales_base_datos_2005_2009_dbf.zip"},
    "2010_2014": {"years": [2010, 2011, 2012, 2013, 2014], "url": "https://www.inegi.org.mx/contenidos/programas/edr/microdatos/defunciones/datos/defunciones_generales_base_datos_2010_2014_dbf.zip"},
    "2015_2019": {"years": [2015, 2016, 2017, 2018, 2019], "url": "https://www.inegi.org.mx/contenidos/programas/edr/microdatos/defunciones/datos/defunciones_generales_base_datos_2015_2019_dbf.zip"},
    "2020":      {"years": [2020], "url": "https://www.inegi.org.mx/contenidos/programas/edr/microdatos/defunciones/2020/defunciones_base_datos_2020_dbf.zip"},
    "2021":      {"years": [2021], "url": "https://www.inegi.org.mx/contenidos/programas/edr/microdatos/defunciones/2021/defunciones_base_datos_2021_dbf.zip"},
    "2022":      {"years": [2022], "url": "https://www.inegi.org.mx/contenidos/programas/edr/microdatos/defunciones/2022/defunciones_base_datos_2022_dbf.zip"},
    "2023":      {"years": [2023], "url": "https://www.inegi.org.mx/contenidos/programas/edr/microdatos/defunciones/2023/defunciones_base_datos_2023_dbf.zip"},
    "2024":      {"years": [2024], "url": "https://www.inegi.org.mx/contenidos/programas/edr/microdatos/defunciones/2024/defunciones_base_datos_2024_dbf.zip"},
}

# NOTA: los archivos son .zip que contienen .dbf (formato DBF), no CSV -- el notebook
# ya soporta ambos formatos en la sección de carga (paso 3).
#
# URLs corregidas por el usuario (2022, 2023, 2024 ya apuntan a su propia carpeta de año).

# Diccionarios de datos (codebooks) por archivo — opcional pero muy recomendado,
# ya que el microdato viene con códigos numéricos (sexo, entidad, causa CIE-10, etc.)
# y los catálogos pueden cambiar entre periodos consolidados.
FILE_GROUP_DICTIONARY_URLS = {
    key: "" for key in FILE_GROUPS
}

# Años que de verdad necesitamos conservar al final (réplica 2008-2018 + extensión 2019-2024)
YEARS_OF_INTEREST = list(range(2008, 2025))

missing = [k for k, v in FILE_GROUPS.items() if not v["url"]]
if missing:
    print(f"⚠️ Faltan URLs para los archivos: {missing}. Complétalas antes de continuar.")
else:
    print("✅ Todas las URLs de datos están completas.")

## 1. Descarga de archivos

In [ ]:
def download_file(url: str, dest_path: Path, chunk_size: int = 1024 * 1024) -> Path:
    """Descarga un archivo por streaming (evita cargarlo completo en memoria)."""
    if dest_path.exists():
        print(f"Ya existe, se omite descarga: {dest_path.name}")
        return dest_path

    print(f"Descargando: {url}")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    print(f"  -> Guardado en: {dest_path} ({dest_path.stat().st_size / 1e6:.1f} MB)")
    return dest_path


downloaded_zips = {}
for group_key, info in FILE_GROUPS.items():
    url = info["url"]
    if not url:
        print(f"[{group_key}] Sin URL, se omite.")
        continue
    dest = RAW_ZIPS_DIR / f"defunciones_{group_key}.zip"
    try:
        downloaded_zips[group_key] = download_file(url, dest)
    except Exception as e:
        print(f"[{group_key}] ERROR al descargar: {e}")

## 2. Extracción de los .zip

In [ ]:
def extract_zip(zip_path: Path, dest_dir: Path) -> list[Path]:
    """Extrae un zip y regresa la lista de archivos extraídos."""
    group_dir = dest_dir / zip_path.stem
    group_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(group_dir)
    extracted = list(group_dir.rglob("*"))
    extracted = [p for p in extracted if p.is_file()]
    return extracted


extracted_files_by_group = {}
for group_key, zip_path in downloaded_zips.items():
    try:
        files = extract_zip(zip_path, RAW_EXTRACTED_DIR)
        extracted_files_by_group[group_key] = files
        print(f"[{group_key}] Archivos extraídos:")
        for f in files:
            print(f"   - {f.name} ({f.suffix})")
    except Exception as e:
        print(f"[{group_key}] ERROR al extraer: {e}")

## 3. Carga de cada archivo a un DataFrame

INEGI ha cambiado el formato de entrega entre periodos (CSV en algunos, DBF en otros). Esta celda intenta detectar automáticamente el archivo de datos principal (excluye diccionarios/catálogos) y lo carga con el lector correcto.

**Nota:** si `dbfread` no está instalado y necesitas leer un `.dbf`, instala con `pip install dbfread --break-system-packages` (o `pip install simpledbf`).

In [ ]:
def find_all_data_files(files: list[Path]) -> list[Path]:
    """Encuentra TODOS los archivos de datos (no solo el más grande).

    IMPORTANTE: se descubrió que los .zip consolidados de INEGI (2005-2009,
    2010-2014, 2015-2019) contienen un archivo .dbf POR CADA AÑO, no un solo
    archivo con todos los años juntos. La versión anterior de esta función
    solo tomaba el archivo más grande (max por tamaño), lo que silenciosamente
    descartaba 4 de los 5 años en cada grupo consolidado. Ahora se cargan y
    concatenan TODOS los archivos de datos encontrados.
    """
    candidates = [
        f for f in files
        if f.suffix.lower() in (".csv", ".dbf")
        and "catalogo" not in f.name.lower()
        and "diccionario" not in f.name.lower()
        and "dic_" not in f.name.lower()
        # Excluye catálogos auxiliares conocidos (no son la tabla de microdatos principal)
        and not any(cat in f.name.upper() for cat in [
            "CATEMLDE", "CATMINDE", "LISTAMEX", "LISTA1", "CAPGPO",
            "GPOLIMEX", "PARENTESCO", "PAISES", "OCUPACIONES", "COD_ADICIO", "LENGUAS"
        ])
    ]
    return candidates


# --- Armonización de tipos de dato entre años ---
# Se descubrió (al intentar guardar en parquet) que INEGI ha cambiado el tipo de
# almacenamiento de varios campos "código" entre años: algunos años los guardan
# como texto (ej. "01") y otros como número (ej. 1) para el MISMO campo lógico
# (ej. Lengua, Nacionalidad, Necropsia, Ocupacion). Al concatenar años con tipos
# distintos, pandas deja la columna como "object" con una mezcla real de str e
# int, lo cual pyarrow no puede convertir a un tipo único de parquet.
#
# Solución: se define explícitamente qué columnas son numéricas "de verdad"
# (medidas o fechas) y todo lo demás se trata como código/categoría → texto,
# rellenando con ceros a la izquierda las columnas de clave geográfica/causa
# que sabemos tienen un ancho fijo (según el diccionario de datos de INEGI),
# sin tocar valores ya alfanuméricos (ej. causas CIE-10 como "X70").

NUMERIC_COLUMNS = {
    'EDAD', 'SEM_GEST', 'GRAMOS', 'HORAS', 'MINUTOS',
    'DIA_OCURR', 'MES_OCURR', 'ANIO_OCUR', 'DIA_REGIS', 'MES_REGIS', 'ANIO_REGIS',
    'DIA_NACIM', 'MES_NACIM', 'ANIO_NACIM', 'DIA_CERT', 'MES_CERT', 'ANIO_CERT',
    'TLOC_REGIS', 'TLOC_RESID', 'TLOC_OCURR', 'CAPITULO', 'GRUPO', 'PAR_AGRE',
    'SEXO', 'EDO_CIVIL', 'ESCOLARIDA', 'PRESUNTO', 'TIPO_DEFUN', 'OCURR_TRAB',
    'LUGAR_OCUR', 'NECROPSIA', 'ASIST_MEDI', 'SITIO_OCUR', 'COND_CERT',
    'NACIONALID', 'DERECHOHAB', 'EMBARAZO', 'REL_EMBA', 'VIO_FAMI', 'AREA_UR',
    'COMPLICARO', 'LENGUA', 'COND_ACT', 'RAZON_M', 'NATVIOLE', 'CIRUGIA',
    'USONECROPS', 'ENCEFALICA', 'DONADOR', 'AFROMEX', 'CONINDIG',
}

# Ancho documentado (según diccionario de datos de INEGI) para columnas de
# clave/código que deben conservar ceros a la izquierda.
CODE_COLUMN_WIDTHS = {
    'ENT_REGIS': 2, 'MUN_REGIS': 3, 'LOC_REGIS': 4,
    'ENT_RESID': 2, 'MUN_RESID': 3, 'LOC_RESID': 4,
    'ENT_OCURR': 2, 'MUN_OCURR': 3, 'LOC_OCURR': 4,
    'ENT_OCULES': 2, 'MUN_OCULES': 3, 'LOC_OCULES': 4,
    'ENT_NAC': 3, 'CAUSA_DEF': 4, 'COD_ADICIO': 4, 'LISTA_MEX': 3,
    'LISTA1': 3, 'GR_LISMEX': 3, 'CVE_LENGUA': 4, 'NACESP_CVE': 3,
    'DIS_RE_OAX': 3, 'MATERNAS': 4, 'OCUPACION': 3,
}


def harmonize_dtypes(df_part: pd.DataFrame) -> pd.DataFrame:
    for col in df_part.columns:
        if col in NUMERIC_COLUMNS:
            df_part[col] = pd.to_numeric(df_part[col], errors="coerce")
        elif col in ("archivo_fuente",):
            continue
        else:
            s = df_part[col].astype("string")
            width = CODE_COLUMN_WIDTHS.get(col)
            if width:
                is_numeric_like = s.str.fullmatch(r"\d+")
                s = s.mask(is_numeric_like.fillna(False), s.str.zfill(width))
            df_part[col] = s
    return df_part


def load_group_dataframe(group_key: str, files: list[Path]) -> pd.DataFrame | None:
    data_files = find_all_data_files(files)
    if not data_files:
        print(f"[{group_key}] No se encontraron archivos de datos principales.")
        return None

    print(f"[{group_key}] Archivos de datos detectados: {[f.name for f in data_files]}")

    dfs = []
    for data_file in data_files:
        print(f"[{group_key}]   Cargando: {data_file.name}")
        if data_file.suffix.lower() == ".csv":
            try:
                df_part = pd.read_csv(data_file, encoding="utf-8", low_memory=False)
            except UnicodeDecodeError:
                df_part = pd.read_csv(data_file, encoding="latin-1", low_memory=False)
        elif data_file.suffix.lower() == ".dbf":
            from dbfread import DBF
            table = DBF(data_file, encoding="latin-1")
            df_part = pd.DataFrame(iter(table))
        else:
            print(f"[{group_key}]   Formato no soportado: {data_file.suffix}, se omite.")
            continue
        df_part["archivo_fuente"] = data_file.name
        df_part = harmonize_dtypes(df_part)
        print(f"[{group_key}]     -> {len(df_part):,} filas")
        dfs.append(df_part)

    if not dfs:
        return None

    df = pd.concat(dfs, ignore_index=True)
    print(f"[{group_key}] Total combinado: {len(df):,} filas | {len(df.columns)} columnas")
    return df


group_dataframes = {}
for group_key, files in extracted_files_by_group.items():
    df_group = load_group_dataframe(group_key, files)
    if df_group is not None:
        group_dataframes[group_key] = df_group

## 4. Revisión de columnas por archivo (para detectar diferencias entre periodos)

Antes de separar por año, revisamos qué tan parecidos son los nombres de columnas entre los distintos archivos consolidados/individuales. Es normal que haya diferencias — las documentaremos en `docs/METHODOLOGY.md` antes de estandarizar.

**Importante:** en esta celda también buscamos, a ojo, cuál columna representa el **año** dentro de cada archivo (ej. `ANIO_OCUR`, `anio_ocur`, `ANIO_OCURR`, etc. — el nombre exacto varía). Anota el nombre correcto para cada `group_key`; lo usamos en la siguiente celda para separar los archivos consolidados en años individuales.

## 4.5 — Diagnóstico: ¿por qué los años anteriores al último de cada grupo tienen tan pocas filas?

Si al correr la sección 5 ves que, dentro de cada archivo consolidado (2005-2009, 2010-2014, 2015-2019), solo el **último año** tiene un número de filas realista (cientos de miles) y los años anteriores tienen apenas cientos o miles, algo no cuadra: no es plausible que México tenga solo 231 defunciones registradas en todo 2010.

Esta celda diagnostica dos posibles causas antes de tocar el filtro:
1. **Tipo de dato inesperado** en la columna de año (ej. vino como texto con espacios, como float con decimales, o como bytes en vez de string — común al leer .dbf con `dbfread`).
2. **La distribución real ya viene así en el archivo fuente** (lo cual sería un hallazgo real sobre cómo INEGI arma estos consolidados, no un bug nuestro).

Corre esta celda y comparte la salida completa antes de decidir cómo corregir el filtro.

In [ ]:
for group_key, df_group in group_dataframes.items():
    year_col = YEAR_COLUMN_BY_GROUP.get(group_key, "")
    if year_col not in df_group.columns:
        continue
    print(f"=== {group_key} ===")
    print("dtype:", df_group[year_col].dtype)
    print("Valores únicos (primeros 20, sin filtrar):")
    print(sorted(df_group[year_col].unique())[:20])
    print("Conteo por valor (top 10 más frecuentes):")
    print(df_group[year_col].value_counts().head(10))
    print("Total de filas en el archivo:", len(df_group))
    print()

In [ ]:
for group_key, df_group in group_dataframes.items():
    print(f"=== {group_key} ===")
    print(list(df_group.columns))
    print()

## 5. Separar cada archivo en años individuales

⚠️ **Completa `YEAR_COLUMN_BY_GROUP`** con el nombre real de la columna de año que identificaste en la celda anterior, para cada `group_key`. Para los archivos que ya son de un solo año (2020-2024) no es estrictamente necesario separar, pero igual filtramos por consistencia.

In [ ]:
# Confirmado con las columnas reales (compartidas por el usuario): la columna de año
# de ocurrencia se llama "ANIO_OCUR" en TODOS los periodos, del 2005-2009 al 2024. No hubo
# que adivinar nada distinto por grupo.
YEAR_COLUMN_BY_GROUP = {
    "2005_2009": "ANIO_OCUR",
    "2010_2014": "ANIO_OCUR",
    "2015_2019": "ANIO_OCUR",
    "2020": "ANIO_OCUR",
    "2021": "ANIO_OCUR",
    "2022": "ANIO_OCUR",
    "2023": "ANIO_OCUR",
    "2024": "ANIO_OCUR",
}

year_dataframes = {}

for group_key, df_group in group_dataframes.items():
    year_col = YEAR_COLUMN_BY_GROUP.get(group_key, "")
    years_in_group = FILE_GROUPS[group_key]["years"]

    if not year_col:
        print(f"[{group_key}] ⚠️ Falta especificar la columna de año, se omite separación.")
        continue

    if year_col not in df_group.columns:
        print(f"[{group_key}] ⚠️ La columna '{year_col}' no existe en este archivo. Columnas disponibles arriba.")
        continue

    for year in years_in_group:
        if year not in YEARS_OF_INTEREST:
            continue
        df_year = df_group[df_group[year_col].astype(str).str.strip() == str(year)].copy()
        df_year["anio_archivo_origen"] = group_key
        year_dataframes[year] = df_year
        print(f"[{group_key}] Año {year}: {len(df_year):,} filas")

## 6. Guardar el consolidado crudo por año (sin filtrar por causa todavía)

Guardamos cada año como está, uno por uno. La estandarización real de nombres de columnas entre periodos y el filtrado a Chihuahua + CIE-10 X60-X84 se documentará y ejecutará en el siguiente notebook/script de limpieza (Etapa 2), una vez que revisemos juntas la salida de las celdas anteriores.

In [ ]:
for year, df_year in sorted(year_dataframes.items()):
    out_path = INTERIM_DIR / f"defunciones_{year}_crudo.parquet"
    df_year.to_parquet(out_path, index=False)
    print(f"Guardado: {out_path} ({len(df_year):,} filas)")

print()
print("Listo. Revisa la salida de las celdas de columnas (secciones 4 y 5) antes de continuar")
print("con la estandarización y el filtrado en el siguiente notebook.")